# Applying LoRA
## Introduction
Low-rank adaptation (LoRA) is a parameter-efficient fine-tuning technique that allows us to adapt large pretrained models to specific tasks with a substantial reduction in computational and memory costs. Instead of adjusting all model parameters, LoRA applies low-rank matrix modifications to key layers, such as attention heads, which means only a small subset of parameters needs to be fine-tuned. This method makes LoRA ideal for adapting large models to task-specific data without the significant resource demands of full model fine-tuning. In this reading, we’ll examine how LoRA functions, the steps for implementing it, and its benefits for fine-tuning large language models efficiently.

By the end of this reading, you will be able to:
* Describe the key concepts and benefits of low-rank adaptation (LoRA) in fine-tuning large models.
* Apply LoRA to a pretrained model for task-specific fine-tuning.
* Fine-tune a model using LoRA with minimized computational and memory resources.
* Evaluate and optimize the performance of a LoRA-fine-tuned model.

## Why use LoRA?
Traditional fine-tuning methods require adjusting all the parameters in a model, which is resource-intensive, especially for large transformer-based models like BERT, RoBERTa, and GPT. As models grow larger, the computational and memory costs of full fine-tuning increase substantially. LoRA addresses these challenges by applying low-rank adaptations within specific layers, focusing on fine-tuning only a subset of parameters that represent a low-rank approximation of the original model's weight matrices. The benefits of LoRA include the following:
* Reduced memory usage: LoRA drastically reduces the memory footprint by fine-tuning only low-rank matrices rather than all model parameters, making it ideal for environments with limited memory capacity.
* Lower computational cost: since fewer parameters are being optimized, LoRA requires less computation, reducing both time and energy consumption.
* Faster training and experimentation: with fewer parameters to update, LoRA shortens training time, enabling faster experimentation and quicker iterations for model improvement.

LoRA is particularly advantageous when working with large models in environments with constrained resources, such as edge devices or research environments in which computational budgets are limited. It also makes fine-tuning large models more feasible for a broader range of applications without requiring access to powerful hardware.

## Step-by-step process to fine-tune a model using LoRA
The remainder of this reading will guide you through the following steps:
* Step 1: Prepare your dataset.
* Step 2: Apply LoRA to the model.
* Step 3: Fine-tune the model with LoRA.
* Step 4: Evaluate the LoRA-fine-tuned model.
* Step 5: Optimize LoRA for your task.

### Step 1: Prepare your dataset
Before you can fine-tune a model using LoRA, it’s essential to ensure that your dataset is preprocessed and structured correctly. Proper dataset preparation is key to achieving reliable performance during fine-tuning and evaluation.

Instructions
* Clean and preprocess the data: remove irrelevant entries, handle missing values, and standardize the text as needed to ensure the data is ready for processing.
* Tokenize the data: use a tokenizer compatible with your chosen model (e.g., a BERT tokenizer for BERT models). This step prepares the text for input into the model.
* Split the dataset: divide the dataset into training, validation, and test sets to allow for reliable performance evaluation. A typical split is 70 percent for training, 15 percent for validation, and 15 percent for testing.

By preparing the dataset carefully, you enable efficient fine-tuning and ensure that your model has access to high-quality, representative data for learning task-specific patterns.

### Step 2: Apply LoRA to the model
Once you have prepared your dataset, you can modify specific layers of a pretrained model using LoRA. The goal is to introduce low-rank matrices to key layers, often the attention layers in transformer models. This modification allows you to fine-tune only the parameters of the low-rank matrices while keeping the rest of the model frozen, significantly reducing computational requirements.

#### Instructions for preparation
* Ensure dataset readiness: confirm that you have preprocessed and tokenized the dataset as outlined in Step 1.
* Understand the model’s architecture: review the structure of the model you’re working with, typically a transformer such as BERT or GPT, to identify layers where you can apply LoRA.
* Identify relevant layers: in transformer-based models, attention layers are often the primary targets for LoRA because they manage most of the information flow in these architectures. By printing out the model’s named modules, you can identify the specific attention layers where LoRA can be introduced. These layers typically have "attention" in their names.

#### Approach
* Load the pretrained model: start with a pretrained model such as BERT to leverage its existing language understanding capabilities.
* Apply LoRA to attention layers: use a LoRA-specific function, such as LoRALayer, to modify only the attention layers.
* Freeze remaining parameters: freeze all other parameters in the model to ensure that only the LoRA-modified layers are adjusted during training.

#### Explanation
* print(name): prints each model component to help locate the attention layers where LoRA can be applied.
* module.apply(LoRALayer): applies the LoRA modification to the identified attention layers.
* param.requires_grad = False: ensures all other parameters remain frozen, meaning only LoRA-modified layers will be fine-tuned.

This setup enables a targeted fine-tuning approach, in which only specific, low-rank parameters are adjusted, minimizing resource use.

In [4]:
from transformers import BertForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

# Load a pre-trained BERT model for classification tasks
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

# Configure and apply LoRA to BERT attention projections
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


### Step 3: Fine-tune the model with LoRA
With LoRA applied to specific layers, you’re ready to fine-tune the model on your task-specific dataset. The goal is to update only the low-rank matrices in the attention layers, optimizing them for the task while keeping the rest of the model’s parameters static.

#### Approach
* Start training: fine-tune the model using the prepared dataset from Step 1.
* Monitor progress: use the validation dataset to track the model’s performance during training.
* Focus on LoRA layers: since LoRA was applied to the attention layers, only the low-rank matrices in these layers will be updated during training, reducing overall computational demand.

#### Explanation
* TrainingArguments(...): specifies key training parameters, such as the number of epochs, batch size, and evaluation frequency.
* Trainer(...): initializes the trainer, linking it to the model, training arguments, and datasets.
* trainer.train(): starts the fine-tuning process, which updates only LoRA-modified layers.

By focusing on just the low-rank matrices, you achieve efficient task-specific fine-tuning without the overhead of updating the entire model.

In [5]:
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

# Fallback datasets (used only if train_data / val_data were not created earlier)
class DummyClassificationDataset(Dataset):
    def __init__(self, n_samples=64, seq_len=32, num_labels=3, vocab_size=30522):
        self.input_ids = torch.randint(0, vocab_size, (n_samples, seq_len))
        self.attention_mask = torch.ones((n_samples, seq_len), dtype=torch.long)
        self.labels = torch.randint(0, num_labels, (n_samples,), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

train_data = globals().get("train_data")
if train_data is None:
    train_data = DummyClassificationDataset(n_samples=128, num_labels=3)

val_data = globals().get("val_data")
if val_data is None:
    val_data = DummyClassificationDataset(n_samples=32, num_labels=3)

# Configure training parameters
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
)

# Set up the Trainer to handle fine-tuning
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
)

# Begin training
trainer.train()

/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,1.157688
2,No log,1.146611
3,No log,1.142482


/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=24, training_loss=1.1275568803151448, metrics={'train_runtime': 7.9912, 'train_samples_per_second': 48.053, 'train_steps_per_second': 3.003, 'total_flos': 6336635387904.0, 'train_loss': 1.1275568803151448, 'epoch': 3.0})

### Step 4: Evaluate the LoRA-fine-tuned model
After fine-tuning, evaluate the model’s performance using standard metrics such as accuracy, F1 score, and precision/recall. Since LoRA optimizes only a small subset of parameters, memory and computational costs are reduced, yet the model can still deliver performance that rivals traditional fine-tuning.

#### Explanation
* trainer.evaluate(...): runs an evaluation on the test dataset.
* results['eval_accuracy']: retrieves the test accuracy, indicating how well the model generalizes to unseen data.

This evaluation step confirms the model’s effectiveness and highlights the efficiency gains from fine-tuning only low-rank matrices, which helps maintain strong performance despite reduced computational overhead.

In [7]:
from transformers.utils.notebook import NotebookProgressCallback

# Evaluate the LoRA fine-tuned model on the test set
test_data = globals().get("test_data")
if test_data is None:
    test_data = DummyClassificationDataset(n_samples=32, num_labels=3)

# Prevent notebook callback error when evaluate() is run independently
trainer.remove_callback(NotebookProgressCallback)

results = trainer.evaluate(eval_dataset=test_data)
print("Evaluation results:", results)

Evaluation results: {'eval_loss': 1.12367844581604, 'eval_runtime': 0.1555, 'eval_samples_per_second': 205.842, 'eval_steps_per_second': 25.73, 'epoch': 3.0}


### Step 5: Optimize LoRA for your task
To achieve even better results, consider experimenting with the rank of the low-rank matrices in LoRA. By adjusting the rank, you can control the number of parameters in the low-rank matrices, balancing the trade-off between computational efficiency and model performance. A higher rank can capture more complexity but may require additional resources, while a lower rank further reduces resource demands.

#### Optimization ideas
* Adjust the rank: experiment with different ranks in the low-rank matrices to find an optimal balance for your specific task.
* Extend LoRA application: apply LoRA to additional layers to capture more complex task-specific features.

#Example of adjusting the rank in LoRA
from lora import adjust_lora_rank

#Set a lower rank for fine-tuning, experiment with values for optimal performance
adjust_lora_rank(model, rank=2)
Explanation
adjust_lora_rank(model, rank=2): sets a lower rank for LoRA, which further reduces the number of parameters involved in fine-tuning, allowing for experiments with different ranks to optimize performance.

This fine-tuning adjustment enables you to fine-tune LoRA-modified layers more precisely, helping the model balance resource use with performance more effectively.

## Conclusion
LoRA provides a resource-efficient alternative to traditional full model fine-tuning, allowing large pretrained models to be tailored to specific tasks with a fraction of the computational cost. By fine-tuning only low-rank approximations within key layers, LoRA enables significant reductions in memory and computational demands while retaining effective performance. This technique is particularly valuable for applications in resource-constrained environments or when experimenting with large models on specialized tasks. By following this guide, you have learned how to apply LoRA to fine-tune models efficiently, making it feasible to leverage powerful language models in various real-world applications without the prohibitive resource requirements typically associated with full fine-tuning.